# Análises Northwind

Ticket médio, itens por pedido, desconto, cross-sell e produtos âncora.

**Pré-requisitos:**
- Postgres ativo (`2-local_setup/start-postgres.ps1`)
- Modelos dbt rodados (`3-dbt/dbt_northwind/run-dbt.ps1 run`)
- Kernel Python: `2-local_setup/.venv` (mesmo ambiente do projeto)

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from sqlalchemy import create_engine, text

EXPECTED_PYTHON = (Path("..") / "2-local_setup" / ".venv" / "Scripts" / "python.exe").resolve()
if Path(sys.executable).resolve() != EXPECTED_PYTHON:
    raise RuntimeError(
        f"Kernel incorreto.\n"
        f"  Atual:     {sys.executable}\n"
        f"  Esperado:  {EXPECTED_PYTHON}\n"
        f"Selecione o kernel 'Python 3 (2-local_setup)' no notebook."
    )

REPORT_DIR = Path(".").resolve()
if not (REPORT_DIR / "sql").exists():
    REPORT_DIR = REPORT_DIR / "5-bi_report"

SQL_DIR = REPORT_DIR / "sql"
OUTPUT_DIR = REPORT_DIR / "output"
ENV_FILE = REPORT_DIR.parent / "2-local_setup" / ".env"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({"figure.figsize": (10, 6), "axes.titlesize": 13, "axes.labelsize": 11})
%matplotlib inline

load_dotenv(ENV_FILE)
engine = create_engine(
    f"postgresql+psycopg2://{os.environ['DBT_USER']}:{os.environ['DBT_PASSWORD']}"
    f"@{os.environ['DBT_HOST']}:{os.environ['DBT_PORT']}/{os.environ['DBT_DATABASE']}"
)

def run_sql(filename: str) -> pd.DataFrame:
    query = (SQL_DIR / filename).read_text(encoding="utf-8")
    return pd.read_sql(text(query), engine)

print(f"Conectado: {engine.url.render_as_string(hide_password=True)}")
print(f"Pasta de saída: {OUTPUT_DIR}")

## 1. Ticket médio por pedido

Receita líquida agregada por pedido e ticket médio geral/mensal.

In [ ]:
df_ticket = run_sql("01_ticket_medio.sql")
df_ticket.to_csv(OUTPUT_DIR / "01_ticket_medio.csv", index=False, encoding="utf-8-sig")
display(df_ticket)

mensal = df_ticket[df_ticket["nivel"] == "mensal"].copy()
geral = df_ticket[df_ticket["nivel"] == "geral"].iloc[0]

fig, ax = plt.subplots()
ax.plot(pd.to_datetime(mensal["mes_pedido"]), mensal["ticket_medio"], marker="o", linewidth=2, color="#2563eb")
ax.axhline(geral["ticket_medio"], color="#dc2626", linestyle="--", label=f"Geral: R$ {geral['ticket_medio']:,.2f}")
ax.set_title("Ticket médio por pedido — evolução mensal")
ax.set_xlabel("Mês")
ax.set_ylabel("Ticket médio (R$)")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "01_ticket_medio_mensal.png", dpi=150)
plt.show()

## 2. Itens por pedido e quantidade média

Quantos itens e unidades cada pedido leva em média.

In [ ]:
df_itens = run_sql("02_itens_por_pedido.sql")
df_itens.to_csv(OUTPUT_DIR / "02_itens_por_pedido.csv", index=False, encoding="utf-8-sig")
display(df_itens)

mensal = df_itens[df_itens["nivel"] == "mensal"].copy()
x = pd.to_datetime(mensal["mes_pedido"])

fig, ax1 = plt.subplots()
ax2 = ax1.twinx()
ax1.bar(x, mensal["media_itens_por_pedido"], width=20, alpha=0.7, color="#7c3aed")
ax2.plot(x, mensal["media_unidades_por_pedido"], color="#059669", marker="s", linewidth=2)
ax1.set_title("Itens e unidades médias por pedido")
ax1.set_xlabel("Mês")
ax1.set_ylabel("Média de itens por pedido")
ax2.set_ylabel("Média de unidades por pedido")
fig.autofmt_xdate()
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "02_itens_unidades_por_pedido.png", dpi=150)
plt.show()

## 3. Impacto do desconto no ticket

Compara ticket médio e participação de pedidos com vs sem desconto.

In [ ]:
df_desconto = run_sql("03_impacto_desconto.sql")
df_desconto.to_csv(OUTPUT_DIR / "03_impacto_desconto.csv", index=False, encoding="utf-8-sig")
display(df_desconto)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].bar(df_desconto["tipo_pedido"], df_desconto["ticket_medio"], color=["#16a34a", "#f97316"])
axes[0].set_title("Ticket médio por tipo de pedido")
axes[0].set_ylabel("Ticket médio (R$)")
axes[1].pie(df_desconto["qtd_pedidos"], labels=df_desconto["tipo_pedido"], autopct="%1.1f%%", colors=["#16a34a", "#f97316"], startangle=90)
axes[1].set_title("Distribuição de pedidos")
fig.suptitle("Impacto do desconto no ticket", fontsize=14)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "03_impacto_desconto.png", dpi=150)
plt.show()

## 4. Cross-sell — categorias que aparecem juntas

Pares de categorias comprados no mesmo pedido.

In [ ]:
df_cross = run_sql("04_cross_sell_categorias.sql")
df_cross.to_csv(OUTPUT_DIR / "04_cross_sell.csv", index=False, encoding="utf-8-sig")
display(df_cross)

top = df_cross.head(10).copy()
top["par_categorias"] = top["categoria_a"] + " + " + top["categoria_b"]

fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top["par_categorias"], top["qtd_pedidos_juntos"], color="#0ea5e9")
ax.invert_yaxis()
ax.set_title("Top pares de categorias no mesmo pedido")
ax.set_xlabel("Quantidade de pedidos")
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "04_cross_sell_categorias.png", dpi=150)
plt.show()

## 5. Produtos âncora vs produtos de expansão

- **Âncora:** produtos no primeiro pedido do cliente
- **Expansão:** produtos em pedidos subsequentes

In [ ]:
df_ancora = run_sql("05_produtos_ancora_expansao.sql")
df_ancora.to_csv(OUTPUT_DIR / "05_ancora_expansao.csv", index=False, encoding="utf-8-sig")
display(df_ancora.head(15))

resumo = df_ancora.groupby("tipo_produto", as_index=False).agg(
    receita_total=("receita_total", "sum"),
    qtd_produtos=("id_produto", "nunique"),
)
top_ancora = df_ancora[df_ancora["tipo_produto"] == "ancora"].head(8)
top_expansao = df_ancora[df_ancora["tipo_produto"] == "expansao"].head(8)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].bar(resumo["tipo_produto"], resumo["receita_total"], color=["#6366f1", "#f59e0b"])
axes[0, 0].set_title("Receita total por tipo")
axes[0, 1].bar(resumo["tipo_produto"], resumo["qtd_produtos"], color=["#6366f1", "#f59e0b"])
axes[0, 1].set_title("Produtos distintos por tipo")
axes[1, 0].barh(top_ancora["nome_produto"], top_ancora["receita_total"], color="#6366f1")
axes[1, 0].invert_yaxis()
axes[1, 0].set_title("Top produtos âncora")
axes[1, 1].barh(top_expansao["nome_produto"], top_expansao["receita_total"], color="#f59e0b")
axes[1, 1].invert_yaxis()
axes[1, 1].set_title("Top produtos de expansão")
fig.suptitle("Produtos âncora vs expansão", fontsize=14)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "05_produtos_ancora_expansao.png", dpi=150)
plt.show()